In [ ]:
from pyspark.sql.session import SparkSession
from pyspark.sql.types import IntegerType

def get_spark_session(appName="default"):
    spark = (
        SparkSession
        .builder
        .appName(appName)
        .master("local[*]")
        .config("spark.ui.port", "4142")
        .config("spark.port.maxRetries", "20") # Changed from 0 to 20 to allow port retries
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    return spark


def stop_start(app_name):
    global spark

    # 1. Safely teardown the old session if present
    if 'spark' in globals() and spark is not None:
        try:
            spark.stop()
            print("Old Spark session successfully stopped.")
        except Exception as e:
            print(f"Failed to stop active session cleanly. Error: {e}")
        finally:
            spark = None  # Completely unbind the dead object from global memory
    else:
        print("No active 'spark' session found to stop.")

    # 2. Return the fresh session
    print(f"Starting new Spark session: '{app_name}'...")
    return get_spark_session(app_name)

# Initialize your fresh execution context
spark = stop_start("working_with_different_data_type")


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import Row
import os, shutil

class CreateSampleData:
    database_name = "sujeet_db"

    def cleanUpWarehouse(self, tableName):
        raw_warehouse_path = spark.conf.get("spark.sql.warehouse.dir")
        clean_warehouse_path = raw_warehouse_path.replace("file:", "")
        target_path = os.path.join(clean_warehouse_path, "sujeet_db.db", tableName)
        
        print(f"Configured Warehouse Path: {raw_warehouse_path}")
        print(f"Target Absolute Path:      {target_path}")
        
        if os.path.exists(target_path):
            print(f"Status: Path found! Proceeding with deletion for {tableName}...")
            try:
                shutil.rmtree(target_path)
                print("Success: Directory deleted successfully.")
            except Exception as e:
                print(f"Error: Failed to delete directory. Reason: {e}")
        else:
            print(f"Status: Path does not exist on the local file system for {tableName}.")

    def prepareData(self, spark: SparkSession):
        print("creating sample dataframe")
        simpleData = [
              Row("James", "Sales", 3000),
              Row("Michael", "Sales", 4600),
              Row("Robert", "Sales", 4100),
              Row("Maria", "Finance", 3000),
              Row("James", "Sales", 3000),
              Row("Scott", "Finance", 3300),
              Row("Jen", "Finance", 3900),
              Row("Jeff", "Marketing", 3000),
              Row("Kumar", "Marketing", 2000),
              Row("Saif", "Sales", 4100),
              Row("Daniel", None, None),
              Row("Christian", None, None)
        ]
        return spark.createDataFrame(simpleData, "name string, department string, salary int")

    def createPaymnetTable(self, spark: SparkSession):
        table_name = "payment"
        print("Creating payment table")
        spark.sql(f"""create database if not exists {CreateSampleData.database_name}""") 
        spark.sql(f"""use {CreateSampleData.database_name}""")
        try:
            self.cleanUpWarehouse(tableName = table_name)
            spark.sql(f"""drop table if exists {CreateSampleData.database_name}.{table_name}""")
            spark.sql(f"""CREATE TABLE if not exists {CreateSampleData.database_name}.{table_name} (payment_amount decimal(8,2), payment_date string, store_id int)""")
            spark.sql("""
                INSERT INTO payment VALUES
                (1200.99, '2018-01-18', 1), (189.23, '2018-02-15', 1), (33.43, '2018-03-03', 3),
                (7382.10, '2019-01-11', 2), (382.92, '2019-02-18', 1), (322.34, '2019-03-29', 2),
                (2929.14, '2020-01-03', 2), (499.02, '2020-02-19', 3), (994.11, '2020-03-14', 1),
                (394.93, '2021-01-22', 2), (3332.23, '2021-02-23', 3), (9499.49, '2021-03-10', 3),
                (3002.43, '2018-02-25', 2), (100.99, '2019-03-07', 1), (211.65, '2020-02-02', 1),
                (500.73, '2021-01-06', 3);
            """)
        except Exception as e:
            print("Table data already exist!")
            raw_warehouse_path = spark.conf.get("spark.sql.warehouse.dir")
            clean_warehouse_path = raw_warehouse_path.replace("file:", "")
            spark.sql(f"""alter table {CreateSampleData.database_name}.{table_name} set location '{clean_warehouse_path}' """)

    def createSalesTable(self, spark: SparkSession):
        """Creates a table populated with the exact values used in the math example [2, 4, 4, 4, 6]"""
        table_name = "sales_metrics"
        print("Creating sales table for variance analysis")
        spark.sql(f"""create database if not exists {CreateSampleData.database_name}""") 
        spark.sql(f"""use {CreateSampleData.database_name}""")
        try:
            self.cleanUpWarehouse(tableName = table_name)
            spark.sql(f"""drop table if exists {CreateSampleData.database_name}.{table_name}""")
            spark.sql(f"""CREATE TABLE if not exists {CreateSampleData.database_name}.{table_name} (day_id int, daily_sales decimal(8,2))""")
            spark.sql("""
                INSERT INTO sales_metrics VALUES
                (1, 2.00),
                (2, 4.00),
                (3, 4.00),
                (4, 4.00),
                (5, 6.00);
            """)
            print("Sales metrics table created successfully.")
        except Exception as e:
            print("Sales table data already exist!")
            raw_warehouse_path = spark.conf.get("spark.sql.warehouse.dir")
            clean_warehouse_path = raw_warehouse_path.replace("file:", "")
            spark.sql(f"""alter table {CreateSampleData.database_name}.{table_name} set location '{clean_warehouse_path}' """)
    
    def createSkewedTable(self, spark: SparkSession):
        """Creates a table populated with the exact values used in the skew/kurtosis math example"""
        table_name = "skew_metrics"
        print("Creating skewed data table...")
        spark.sql(f"""CREATE DATABASE IF NOT EXISTS {CreateSampleData.database_name}""") 
        spark.sql(f"""USE {CreateSampleData.database_name}""")
        try:
            self.cleanUpWarehouse(tableName = table_name)
            spark.sql(f"""DROP TABLE IF EXISTS {CreateSampleData.database_name}.{table_name}""")
            spark.sql(f"""CREATE TABLE IF NOT EXISTS {CreateSampleData.database_name}.{table_name} (id INT, item_value DECIMAL(8,2))""")
            spark.sql("""
                INSERT INTO skew_metrics VALUES
                (1, 2.00),
                (2, 2.00),
                (3, 2.00),
                (4, 2.00),
                (5, 12.00);
            """)
            print("Skew metrics table created successfully.")
        except Exception as e:
            print("Skew table update fallback triggered.")
            raw_warehouse_path = spark.conf.get("spark.sql.warehouse.dir").replace("file:", "")
            spark.sql(f"ALTER TABLE {CreateSampleData.database_name}.{table_name} SET LOCATION '{raw_warehouse_path}'")

    def createCorrelationTable(self, spark: SparkSession):
        """Creates a table with paired variables to analyze Covariance and Correlation"""
        table_name = "weather_sales"
        print("Creating table for covariance and correlation analysis...")
        spark.sql(f"""CREATE DATABASE IF NOT EXISTS {CreateSampleData.database_name}""") 
        spark.sql(f"""USE {CreateSampleData.database_name}""")
        try:
            self.cleanUpWarehouse(tableName = table_name)
            spark.sql(f"""DROP TABLE IF EXISTS {CreateSampleData.database_name}.{table_name}""")
            spark.sql(f"""CREATE TABLE IF NOT EXISTS {CreateSampleData.database_name}.{table_name} (day_id INT, temperature DECIMAL(8,2), ice_cream_sales DECIMAL(8,2))""")
            spark.sql("""
                INSERT INTO weather_sales VALUES
                (1, 2.00, 10.00),
                (2, 4.00, 20.00),
                (3, 4.00, 25.00),
                (4, 4.00, 25.00),
                (5, 6.00, 40.00);
            """)
            print("Weather sales table created successfully.")
        except Exception as e:
            print("Correlation table update fallback triggered.")
            raw_warehouse_path = spark.conf.get("spark.sql.warehouse.dir").replace("file:", "")
            spark.sql(f"ALTER TABLE {CreateSampleData.database_name}.{table_name} SET LOCATION '{raw_warehouse_path}'")


obj = CreateSampleData()
obj.createPaymnetTable(spark)
obj.createSalesTable(spark)
obj.createSkewedTable(spark)
obj.createCorrelationTable(spark)


